# Single-Agent Smart Assistant + Quiz Auto-Grader

**Part 1** — single-agent pipeline: conditional routing, calculator tool, keyword-extraction tool, error handling.

**Part 2** — uses the keyword tool built in Part 1 to grade the Week 8 quiz answers against a reference keyword set and produce a score.

## Part 1 — Agent Pipeline

In [1]:
import re
import json
import time
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("agent")


In [2]:

import ast
import operator as op

class CalculatorTool:
    # uses ast instead of eval so it can't blow up on something like 9**9**9**9
    name = "calculator_tool"

    _ops = {
        ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
        ast.Div: op.truediv, ast.USub: op.neg, ast.UAdd: op.pos,
    }

    def _eval_node(self, node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in self._ops:
            return self._ops[type(node.op)](self._eval_node(node.left), self._eval_node(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in self._ops:
            return self._ops[type(node.op)](self._eval_node(node.operand))
        raise ValueError("Unsupported expression")

    def run(self, query: str) -> dict:
        expr_match = re.search(r"[\d\.\s\+\-\*\/\(\)]+", query)
        if not expr_match:
            raise ValueError("No valid math expression found")
        expr = expr_match.group().strip()

        if not re.fullmatch(r"[\d\.\s\+\-\*\/\(\)]+", expr) or "**" in expr:
            raise ValueError("Unsafe expression")

        try:
            tree = ast.parse(expr, mode="eval")
            result = self._eval_node(tree.body)
        except ZeroDivisionError:
            raise ValueError("Division by zero")
        except (SyntaxError, ValueError):
            raise ValueError(f"Could not parse expression: {expr!r}")

        return {"tool": self.name, "expression": expr, "result": result}


STOPWORDS = {
    "the","a","an","is","are","was","were","in","on","of","to","and","or","for",
    "with","that","this","it","as","by","be","from","at","which","how","what",
    "why","when","can","could","would","should","if","then","than","its","their",
    "these","those","also","such","into","over","between","each","other","more",
    "so","do","does","did","not","no","but","has","have","had","will","may",
    "used","use","using","provide","give","example","explain","describe","define",
}

class KeywordExtractionTool:
    name = "keyword_tool"

    def run(self, query: str) -> dict:
        text = query.lower()
        text = re.sub(r"[^a-z0-9\s\-]", " ", text)
        words = [w.strip("-") for w in text.split()]
        words = [w for w in words if w and w not in STOPWORDS and len(w) > 2]

        freq = {}
        for w in words:
            freq[w] = freq.get(w, 0) + 1
        keywords = sorted(set(words), key=lambda w: -freq[w])
        return {"tool": self.name, "keywords": keywords, "count": len(keywords)}


In [3]:

class Agent:
    def __init__(self):
        self.calculator = CalculatorTool()
        self.keyword_tool = KeywordExtractionTool()
        self.max_retries = 3

    def route(self, query: str) -> str:
        q = query.lower()
        has_arithmetic = bool(re.search(r"\d+\s*[\+\-\*\/]\s*\d+", q))
        wants_keywords = "keyword" in q

        # "calculate the keyword frequency" should go to keyword_node, not
        # calculator_node, even though it contains "calculate"
        if wants_keywords and not has_arithmetic:
            return "keyword_node"
        if has_arithmetic or "calculate" in q:
            return "calculator_node"
        if wants_keywords:
            return "keyword_node"
        return "general_node"

    _NON_RETRYABLE = (ValueError,)  # no point retrying errors that'll never change

    def _with_retry(self, fn, *args):
        last_err = None
        for attempt in range(1, self.max_retries + 1):
            try:
                return fn(*args)
            except self._NON_RETRYABLE as e:
                logger.info(f"non-retryable error: {e}")
                return {"error": str(e)}
            except Exception as e:
                last_err = e
                logger.info(f"attempt {attempt} failed: {e}")
                time.sleep(0)
        return {"error": str(last_err)}

    def calculator_node(self, query, state):
        result = self._with_retry(self.calculator.run, query)
        state["result"] = result
        state["path"].append("calculator_node")
        return state

    def keyword_node(self, query, state):
        result = self._with_retry(self.keyword_tool.run, query)
        state["result"] = result
        state["path"].append("keyword_node")
        return state

    def general_node(self, query, state):
        state["result"] = {"tool": "general_response", "response": f"Here is a direct answer for: {query}"}
        state["path"].append("general_node")
        return state

    def run(self, query: str) -> dict:
        state = {"query": query, "path": ["route"], "result": None}
        node = self.route(query)
        handler = getattr(self, node)
        state = handler(query, state)
        return state


agent = Agent()


In [4]:

for q in ["calculate 12 + 8 * 2", "extract keywords from this sentence about agents", "tell me about the weather"]:
    print(q, "->", agent.run(q))


calculate 12 + 8 * 2 -> {'query': 'calculate 12 + 8 * 2', 'path': ['route', 'calculator_node'], 'result': {'tool': 'calculator_tool', 'expression': '12 + 8 * 2', 'result': 28}}
extract keywords from this sentence about agents -> {'query': 'extract keywords from this sentence about agents', 'path': ['route', 'keyword_node'], 'result': {'tool': 'keyword_tool', 'keywords': ['about', 'sentence', 'extract', 'keywords', 'agents'], 'count': 5}}
tell me about the weather -> {'query': 'tell me about the weather', 'path': ['route', 'general_node'], 'result': {'tool': 'general_response', 'response': 'Here is a direct answer for: tell me about the weather'}}


## Part 2 — Quiz Auto-Grader


In [5]:
QUIZ = [
    {
        "id": 1,
        "question": "Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?",
        "student_answer": "A stateful directed graph is a workflow where each step can store and use information from previous steps. The workflow is made up of nodes and edges that define how data moves through the system. It can support branching, looping, and decision-making. This makes it more flexible and intelligent. In contrast, a linear pipeline follows a fixed sequence of steps from start to finish without remembering previous states or changing its path.",
        "reference_keywords": ["state", "nodes", "edges", "branching", "looping", "decision-making", "linear", "fixed sequence"],
    },
    {
        "id": 2,
        "question": "Describe the role of nodes and edges in an agent workflow. Give an example of each.",
        "student_answer": "Nodes are the individual tasks or actions performed in an agent workflow. They can represent operations such as processing a query, calling a tool, or generating a response. Edges are the connections between nodes that determine how information flows. For example, a Calculator Tool can be a node, while the path connecting query analysis to the calculator is an edge. Together, nodes and edges define the complete workflow.",
        "reference_keywords": ["nodes", "tasks", "actions", "edges", "connections", "information flow", "example"],
    },
    {
        "id": 3,
        "question": "What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.",
        "student_answer": "Conditional routing is the process of directing a query to different tools or actions based on its intent. It helps the agent choose the most appropriate response method. For example, if a query contains the word calculate, it is sent to the Calculator Tool. If it contains keywords, it is sent to the Keyword Extraction Tool. All other queries can be handled by a General Response module.",
        "reference_keywords": ["conditional routing", "intent", "calculator tool", "keyword extraction tool", "general response", "rule-based"],
    },
    {
        "id": 4,
        "question": "Why are cycles (loops) important in agent pipelines? Provide a use case where a retry loop is necessary.",
        "student_answer": "Cycles or loops allow an agent to repeat a process until a desired result is achieved. They are useful when tasks may fail or require multiple attempts. For example, if an API request fails due to a temporary network issue, the agent can retry the request instead of immediately returning an error. This improves reliability and robustness. Without loops, the workflow would stop after a single failure.",
        "reference_keywords": ["cycles", "loops", "repeat", "retry", "api request", "failure", "reliability"],
    },
    {
        "id": 5,
        "question": "Explain how a single-agent system can simulate multi-agent behavior internally.",
        "student_answer": "A single-agent system can simulate multiple agents by dividing its work into different roles. It may first analyze the query, then decide which tool to use, and finally generate a response. Each role behaves like a separate agent even though they are all executed by one system. This approach reduces complexity while maintaining flexibility. As a result, a single agent can perform tasks similar to a multi-agent architecture.",
        "reference_keywords": ["roles", "divide", "analyze query", "decide tool", "generate response", "single system", "multi-agent"],
    },
    {
        "id": 6,
        "question": "What are JSON schema tools? How do they help in structuring tool inputs and outputs?",
        "student_answer": "JSON schema tools define a standard structure for data exchanged between agents and tools. They specify required fields, data types, and expected formats. This ensures that inputs are valid before processing begins. It also makes outputs consistent and easier to interpret. Using JSON schemas reduces errors and improves communication between different components of an agent system.",
        "reference_keywords": ["json schema", "standard structure", "required fields", "data types", "validation", "consistent output"],
    },
    {
        "id": 7,
        "question": "Compare sequential tool calls and parallel tool calls. When would you prefer one over the other?",
        "student_answer": "Sequential tool calls are executed one after another, where each step depends on the result of the previous step. Parallel tool calls execute multiple independent tasks at the same time. Sequential execution is preferred when tasks have dependencies. Parallel execution is useful when tasks are unrelated and can be completed simultaneously. Using parallel calls can significantly reduce response time and improve efficiency.",
        "reference_keywords": ["sequential", "parallel", "dependencies", "independent tasks", "response time", "efficiency"],
    },
    {
        "id": 8,
        "question": "How would you implement error handling in a tool-using agent? Provide at least two strategies.",
        "student_answer": "Error handling helps an agent continue operating even when problems occur. One strategy is using try-except blocks to catch exceptions and return meaningful error messages. Another strategy is implementing retry mechanisms that automatically repeat failed operations. Logging errors is also useful for debugging and monitoring. These techniques improve reliability and help maintain a better user experience.",
        "reference_keywords": ["try-except", "exceptions", "retry mechanism", "logging", "debugging", "reliability"],
    },
    {
        "id": 9,
        "question": "What is trajectory evaluation in agent systems? Why is it important beyond just checking final output?",
        "student_answer": "Trajectory evaluation examines the entire sequence of actions taken by an agent to solve a task. It focuses on the decisions, tool calls, and intermediate steps used during execution. This helps identify mistakes that may not be visible in the final answer. It is useful for debugging, optimization, and improving agent performance. By evaluating the full trajectory, developers gain a deeper understanding of agent behavior.",
        "reference_keywords": ["trajectory", "sequence of actions", "decisions", "tool calls", "intermediate steps", "debugging", "optimization"],
    },
    {
        "id": 10,
        "question": "Define task completion rate and cost metrics. How would you measure and optimize them in a real-world system?",
        "student_answer": "Task completion rate measures the percentage of tasks successfully completed by an agent. Cost metrics represent the resources consumed, such as API calls, processing time, or monetary expenses. These metrics can be measured by tracking agent performance over multiple tasks. To optimize them, developers can improve routing accuracy, reduce unnecessary tool calls, and use efficient algorithms.",
        "reference_keywords": ["completion rate", "cost metrics", "resources", "api calls", "processing time", "routing accuracy", "optimize"],
    },
]
print(f"Loaded {len(QUIZ)} questions")


Loaded 10 questions


In [6]:

def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def grade_answer(item: dict, answer_text: str, tool: KeywordExtractionTool) -> dict:
    norm_answer = normalize(answer_text or "")

    if not norm_answer:
        return {
            "id": item["id"],
            "extracted_keywords": [],
            "matched": [],
            "missed": list(item["reference_keywords"]),
            "score": 0.0,
            "flag": "empty_answer",
        }

    extracted = tool.run(answer_text)["keywords"]

    matched, missed = [], []
    for ref in item["reference_keywords"]:
        ref_norm = normalize(ref)
        if ref_norm in norm_answer or all(w in norm_answer for w in ref_norm.split()):
            matched.append(ref)
        else:
            missed.append(ref)

    score = len(matched) / len(item["reference_keywords"])
    return {
        "id": item["id"],
        "extracted_keywords": extracted[:10],
        "matched": matched,
        "missed": missed,
        "score": round(score, 2),
    }


In [7]:

def run_quiz(quiz: list, tool: KeywordExtractionTool, answers: dict = None) -> list:
    # pass answers={id: text} to grade non-interactively (needed for Run All in Colab,
    # since input() just hangs there). leave it out to type answers live like before.
    interactive = answers is None
    results = []
    total = 0
    for item in quiz:
        qid = item["id"]; qtext = item["question"]
        print(f"\nQ{qid}: {qtext}")

        if interactive:
            answer = input("Your answer: ")
        else:
            answer = answers.get(qid, "")
            print(f"Answer: {answer}")

        res = grade_answer(item, answer, tool)
        results.append(res)
        total += res["score"]
        sc = res["score"] * 100; mt = res["matched"]; ms = res["missed"]
        flag = f"  [{res['flag']}]" if "flag" in res else ""
        print(f"Score: {sc:.0f}%  matched={mt}  missed={ms}{flag}")

    avg = total / len(quiz)
    print(f"\nOverall score: {avg*100:.1f}%  ({total:.2f} / {len(quiz)})")
    return results


student_answers = {item["id"]: item["student_answer"] for item in QUIZ}
results = run_quiz(QUIZ, agent.keyword_tool, answers=student_answers)



Q1: Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?
Answer: A stateful directed graph is a workflow where each step can store and use information from previous steps. The workflow is made up of nodes and edges that define how data moves through the system. It can support branching, looping, and decision-making. This makes it more flexible and intelligent. In contrast, a linear pipeline follows a fixed sequence of steps from start to finish without remembering previous states or changing its path.
Score: 100%  matched=['state', 'nodes', 'edges', 'branching', 'looping', 'decision-making', 'linear', 'fixed sequence']  missed=[]

Q2: Describe the role of nodes and edges in an agent workflow. Give an example of each.
Answer: Nodes are the individual tasks or actions performed in an agent workflow. They can represent operations such as processing a query, calling a tool, or generating a response. Edges are the connection